# 06. Atribuir pelo weight (1 CPF por Censo + drop de colisão)

Não altera o greedy do [`04_atribuir.ipynb`](04_atribuir.ipynb). Todos os Censos
do subset. Sem filtro de cluster.

Candidatos: pares do 02b com `match_weight` no corte operacional de
`THRESHOLD_AVALIACAO` (config, hoje 0,99) → `w ≥ log2(T/(1-T))`. A escolha é
sempre por `match_weight` (já no parquet); a probability não entra no SQL.

Passagem 1: por Censo, o CPF de maior weight. Sem par acima do corte → nulo.
Passagem 2: CPF com um único Censo no topo do weight fica com esse; os outros
nulos. Se dois ou mais Censos empatam no **mesmo** weight no topo do CPF,
descarta (ninguém fica com o CPF).

`n_extras` = disputas com vencedor único de weight.

Pré-requisito: 02b (predictions), 00b (limpos), coorte.


In [ ]:
import sys
from math import log2
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from IPython.display import display

from config import (
    COHORT_DEDUP_ARQUIVO,
    METRICAS_PESO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    THRESHOLD_AVALIACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_gt_no_subset,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

TOP_N = 30
WEIGHT_MIN = log2(THRESHOLD_AVALIACAO / (1.0 - THRESHOLD_AVALIACAO))


def display_metricas(df_uma_linha):
    """Uma linha de métricas: inteiros com milhar, o resto com 4 casas."""
    row = df_uma_linha.iloc[0]
    linhas = []
    for k, v in row.items():
        if v is None or (isinstance(v, float) and pd.isna(v)):
            txt = ''
        elif isinstance(v, bool):
            txt = v
        elif pd.api.types.is_number(v) and not isinstance(v, bool):
            fv = float(v)
            if abs(fv - round(fv)) < 1e-12 and abs(fv) >= 1:
                txt = f'{int(round(fv)):,}'
            else:
                txt = f'{fv:.4f}'
        else:
            txt = v
        linhas.append({'metrica': k, 'valor': txt})
    display(pd.DataFrame(linhas))


print_paths()
require_input(COHORT_DEDUP_ARQUIVO, label='COHORT')
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
print('THRESHOLD_AVALIACAO:', THRESHOLD_AVALIACAO)
print('WEIGHT_MIN (bits):', round(WEIGHT_MIN, 6))


## 0. Predictions e ouro

`unique_id_l` = Censo. O parquet precisa ter `match_weight` (export do 02b).
Só entram pares com `match_weight ≥ WEIGHT_MIN` (corte `THRESHOLD_AVALIACAO`).


In [ ]:
cols_pred = con.execute(
    f"SELECT * FROM read_parquet('{SPLINK_PREDICTIONS}') LIMIT 0"
).df().columns
if 'match_weight' not in cols_pred:
    raise RuntimeError(
        f'{SPLINK_PREDICTIONS} não tem match_weight. Rode de novo o 02b_aplicar.'
    )

con.execute(f'''
CREATE OR REPLACE TABLE splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_l,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_r,
    match_weight
FROM read_parquet('{SPLINK_PREDICTIONS}')
WHERE match_weight >= {WEIGHT_MIN}
''')

counts = materialize_gt_no_subset(con, cohort_parquet=COHORT_DEDUP_ARQUIVO)
n_gt = counts['n_gt_no_subset']
print('Ouro 1:1 no subset:', f'{n_gt:,}')
print(
    f'Pares com weight ≥ {WEIGHT_MIN:.4f} '
    f'(THRESHOLD_AVALIACAO={THRESHOLD_AVALIACAO}):',
    con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0],
)


## 1. Duas passagens por weight

Passagem 1: melhor CPF por Censo (só pares no corte). Passagem 2: no CPF,
Censo de maior weight se o topo for único; empate de weight no topo descarta.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE censo_subset AS
SELECT unique_id AS unique_id_censo
FROM {SPLINK_INPUT_VIEW}
WHERE origem = 'censo'
''')

con.execute('''
CREATE OR REPLACE TABLE peso_pass1 AS
SELECT unique_id_censo, unique_id_cpf, match_weight
FROM (
    SELECT
        unique_id_l AS unique_id_censo,
        unique_id_r AS unique_id_cpf,
        match_weight,
        COUNT(*) OVER (
            PARTITION BY unique_id_l, match_weight
        ) AS n_mesmo_w,
        MAX(match_weight) OVER (PARTITION BY unique_id_l) AS wmax
    FROM splink_predictions
)
WHERE match_weight = wmax AND n_mesmo_w = 1
''')

con.execute('''
CREATE OR REPLACE TABLE cpf_topo AS
SELECT
    unique_id_cpf,
    MAX(match_weight) AS wmax,
    COUNT(*) AS n_censo
FROM peso_pass1
GROUP BY 1
''')

con.execute('''
CREATE OR REPLACE TABLE cpf_n_no_topo AS
SELECT p.unique_id_cpf, COUNT(*) AS n_no_topo
FROM peso_pass1 p
JOIN cpf_topo t
  ON t.unique_id_cpf = p.unique_id_cpf
 AND p.match_weight = t.wmax
GROUP BY 1
''')

con.execute('''
CREATE OR REPLACE TABLE atribuicao_peso AS
SELECT
    c.unique_id_censo,
    CASE
        WHEN p.unique_id_cpf IS NULL THEN NULL
        WHEN t.n_censo = 1 THEN p.unique_id_cpf
        WHEN n.n_no_topo = 1 AND p.match_weight = t.wmax THEN p.unique_id_cpf
        ELSE NULL
    END AS unique_id_cpf,
    CASE
        WHEN p.unique_id_cpf IS NULL THEN NULL
        WHEN t.n_censo = 1 THEN p.match_weight
        WHEN n.n_no_topo = 1 AND p.match_weight = t.wmax THEN p.match_weight
        ELSE NULL
    END AS match_weight,
    CASE
        WHEN p.unique_id_cpf IS NULL THEN 'sem_par'
        WHEN t.n_censo = 1 THEN 'unico_pass1'
        WHEN n.n_no_topo > 1 THEN 'nulo_empate'
        WHEN n.n_no_topo = 1 AND p.match_weight = t.wmax THEN 'extra_disputa'
        ELSE 'nulo_pass2'
    END AS status
FROM censo_subset c
LEFT JOIN peso_pass1 p ON p.unique_id_censo = c.unique_id_censo
LEFT JOIN cpf_topo t ON t.unique_id_cpf = p.unique_id_cpf
LEFT JOIN cpf_n_no_topo n ON n.unique_id_cpf = p.unique_id_cpf
''')

metricas = con.execute(f'''
SELECT
    CAST({THRESHOLD_AVALIACAO} AS DOUBLE) AS threshold_avaliacao,
    CAST({WEIGHT_MIN} AS DOUBLE) AS weight_min,
    CAST((SELECT COUNT(*) FROM censo_subset) AS BIGINT) AS n_censo,
    CAST((SELECT COUNT(*) FROM peso_pass1) AS BIGINT) AS n_com_par_pass1,
    CAST((
        SELECT COUNT(*) FROM cpf_topo WHERE n_censo = 1
    ) AS BIGINT) AS n_cpf_unico_pass1,
    CAST((
        SELECT COUNT(*) FROM cpf_topo WHERE n_censo > 1
    ) AS BIGINT) AS n_cpf_disputado,
    CAST((
        SELECT COUNT(*) FROM cpf_n_no_topo n
        JOIN cpf_topo t ON t.unique_id_cpf = n.unique_id_cpf
        WHERE t.n_censo > 1 AND n.n_no_topo = 1
    ) AS BIGINT) AS n_extras,
    CAST((
        SELECT COUNT(*) FROM cpf_n_no_topo WHERE n_no_topo > 1
    ) AS BIGINT) AS n_cpf_empate_descartado,
    CAST((
        SELECT COUNT(*) FROM atribuicao_peso WHERE status = 'nulo_empate'
    ) AS BIGINT) AS n_censo_nulo_empate,
    CAST((
        SELECT COUNT(*) FROM atribuicao_peso WHERE status = 'nulo_pass2'
    ) AS BIGINT) AS n_censo_nulo_pass2,
    CAST((
        SELECT COUNT(*) FROM atribuicao_peso WHERE unique_id_cpf IS NOT NULL
    ) AS BIGINT) AS n_1a1_pass2,
    CAST(COALESCE((
        SELECT SUM(CASE WHEN a.unique_id_cpf = gt.unique_id_cpf THEN 1 ELSE 0 END)
        FROM atribuicao_peso a
        JOIN gt_no_subset gt ON gt.unique_id_censo = a.unique_id_censo
        WHERE a.unique_id_cpf IS NOT NULL
    ), 0) AS BIGINT) AS n_ouro_cpf_certo
''').df()
display_metricas(metricas)

display(con.execute('''
SELECT status, CAST(COUNT(*) AS BIGINT) AS n
FROM atribuicao_peso
GROUP BY status
ORDER BY n DESC
''').df())


## 2. Amostras

Extras (disputa com vencedor único de weight), nulos da passagem 2 e empates descartados.


In [ ]:
attrs = f'''
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.sexo AS sexo_censo,
    pb.sexo AS sexo_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.idade AS idade_censo,
    pb.idade AS idade_cpf
'''

print('Amostra extras (CPF disputado, vencedor único de weight):')
display(con.execute(f'''
SELECT
    a.unique_id_censo, a.unique_id_cpf, a.match_weight, a.status,
    {attrs}
FROM atribuicao_peso a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
WHERE a.status = 'extra_disputa'
ORDER BY a.match_weight DESC
LIMIT {TOP_N}
''').df())

print('Amostra nulos da passagem 2 (perderam o CPF na disputa):')
display(con.execute(f'''
SELECT
    n.unique_id_censo, p1.unique_id_cpf AS unique_id_cpf_perdido,
    p1.match_weight, n.status,
    {attrs}
FROM atribuicao_peso n
JOIN peso_pass1 p1 ON p1.unique_id_censo = n.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = n.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p1.unique_id_cpf
WHERE n.status = 'nulo_pass2'
ORDER BY p1.match_weight DESC
LIMIT {TOP_N}
''').df())

print('Amostra empate de weight no mesmo CPF (descartado):')
display(con.execute(f'''
SELECT
    n.unique_id_censo, p1.unique_id_cpf AS unique_id_cpf_empate,
    p1.match_weight, n.status,
    {attrs}
FROM atribuicao_peso n
JOIN peso_pass1 p1 ON p1.unique_id_censo = n.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = n.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p1.unique_id_cpf
WHERE n.status = 'nulo_empate'
ORDER BY p1.match_weight DESC
LIMIT {TOP_N}
''').df())


In [ ]:
metricas.to_csv(METRICAS_PESO, index=False)
print('Métricas:', METRICAS_PESO)
display_metricas(metricas)
con.close()
